# models

> `EncoderSpec`, `DecisionHead` and `EncoderDecisionModel`: an encoder becomes a decision model through what the checkpoint offers, Laya's head sized to it, and a training regime.

In [ ]:
#| default_exp models

In [ ]:
#| export
import copy, importlib, json, os, re, struct, urllib.request, warnings
from contextlib import contextmanager, nullcontext
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from fastcore.basics import patch, store_attr
from transformers import AutoConfig, AutoModel, AutoTokenizer

from sysone.core import *
from sysone.template import RowTemplate, RowBuilder

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail, test_close, test_ne
import json
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()

Everything downstream of the encoder needs only a hidden state per position and the positions of the anchors. So an encoder is described by a few facts, its width, its context, its family and modality, its special tokens, and the head is sized from them. The tests use the fixture encoder, a random four-layer, 64-wide ModernBERT:

In [ ]:
FIX = "fixtures/tiny-modernbert"
AutoConfig.from_pretrained(FIX)

ModernBertConfig {
  "architectures": [
    "ModernBertModel"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "classifier_activation": "gelu",
  "classifier_bias": false,
  "classifier_dropout": 0.0,
  "classifier_pooling": "cls",
  "cls_token_id": 1,
  "decoder_bias": true,
  "deterministic_flash_attn": false,
  "dtype": "float32",
  "embedding_dropout": 0.0,
  "eos_token_id": 2,
  "global_attn_every_n_layers": 2,
  "hidden_activation": "gelu",
  "hidden_size": 64,
  "initializer_cutoff_factor": 2.0,
  "initializer_range": 0.02,
  "intermediate_size": 96,
  "layer_types": [
    "full_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention"
  ],
  "local_attention": 64,
  "max_position_embeddings": 1024,
  "mlp_bias": false,
  "mlp_dropout": 0.0,
  "model_type": "modernbert",
  "norm_bias": false,
  "norm_eps": 1e-05,
  "num_attention_heads": 2,
  "num_hidden_layers": 4,
  "pad_token_id": 3,
  "rope_parameters": {
    "full_a

## EncoderSpec

`EncoderSpec.from_pretrained` reads the config and the tokenizer (or processor) once. Known families ship their row template and their LoRA targets; any other encoder whose tokenizer defines cls, sep and mask gets Laya's template through the `auto` preset, and anything else must name a `RowTemplate`. Checkpoints that are not plain transformers models (a GLiNER2 checkpoint, whose encoder sits inside it) are recognised by *sources* registered in `ENCODER_SOURCES`; see the GLiNER encoder notebook.

In [ ]:
#| export
# family: (row template preset, LoRA target modules on the attention projections)
FAMILIES = {
    "modernbert": ("laya", r".*\.attn\.(Wqkv|Wo)"),
    "neomme": ("neomme", ["q_proj", "k_proj", "v_proj", "o_proj"]),
    "modernvbert": ("modernvbert", r"text_model\.layers\.\d+\.attn\.(Wqkv|Wo)"),     # the text tower's attention; the image tower stays frozen
    "deberta-v2": ("laya", ["query_proj", "key_proj", "value_proj", "attention.output.dense"]),
    "bert": ("auto", ["query", "key", "value", "attention.output.dense"]),
    "roberta": ("auto", ["query", "key", "value", "attention.output.dense"]),
    "xlm-roberta": ("auto", ["query", "key", "value", "attention.output.dense"]),
    "electra": ("auto", ["query", "key", "value", "attention.output.dense"]),
}
ENCODER_SOURCES = []   # (name, detect(id, revision) -> bool, spec(id, revision, **kw) -> EncoderSpec)

def _text_config(config): return getattr(config, "text_config", None) or config

def _is_multimodal(config):
    return getattr(config, "vision_config", None) is not None or getattr(config, "image_token_id", None) is not None

SOURCE_MODULES = {"protst": "sysone.protein"}   # sources that sysone's own modules register, imported when a spec or an export names them

def ensure_source(source:str|None):
    "Import the sysone module that registers `source`, when it is one of sysone's own"
    if source in SOURCE_MODULES: importlib.import_module(SOURCE_MODULES[source])

class EncoderSpec:
    "What an encoder checkpoint offers: width, context, family, modality, special tokens, default template, side streams"
    def __init__(self, id:str, config, tokenizer, processor=None, revision:str|None=None, source:str="transformers",
                 template:RowTemplate|None=None, loader=None, streams:list|None=None):
        store_attr()
        self.streams = {s.name: s for s in streams or []}
        self.width = _text_config(config).hidden_size
        self.context = getattr(_text_config(config), "max_position_embeddings", None) or 512
        self.family = getattr(config, "model_type", None)
        self.modality = "multimodal" if processor is not None else "text"
        if self.template is None: self.template = self._default_template()

    @classmethod
    def from_pretrained(cls, id:str, revision:str|None=None, source:str|None=None, **kw):
        "The spec of a Hub id or local folder (`source` names a registered source, skipping detection)"
        ensure_source(source)
        for name, detect, make in ENCODER_SOURCES:
            if source is not None and name == source: return make(id, revision=revision, **kw)
        for name, detect, make in ENCODER_SOURCES:
            if detect(id, revision): return make(id, revision=revision, **kw)
        config = AutoConfig.from_pretrained(id, revision=revision)
        processor = None
        if _is_multimodal(config):
            from transformers import AutoProcessor
            processor = AutoProcessor.from_pretrained(id, revision=revision)
        tokenizer = processor.tokenizer if processor is not None else AutoTokenizer.from_pretrained(id, revision=revision)
        return cls(id, config, tokenizer, processor, revision=revision)

    def __repr__(self):
        streams = f", streams {list(self.streams)}" if self.streams else ""
        return (f"EncoderSpec({self.id}: {self.family}, width {self.width}, context {self.context}, {self.modality}, "
                f"template {self.template.name}{streams})")

In [ ]:
#| export
@patch
def _default_template(self:EncoderSpec) -> RowTemplate:
    preset = FAMILIES.get(self.family, ("auto", None))[0]
    t = RowTemplate.preset(preset, tokenizer=self.tokenizer)
    return t.replace(max_len=min(t.max_len, self.context)) if t.max_len > self.context else t

@patch(as_prop=True)
def specials(self:EncoderSpec) -> dict:
    "The tokenizer's special tokens"
    return {r: getattr(self.tokenizer, f"{r}_token", None) for r in ("cls", "sep", "mask", "pad", "unk", "bos", "eos")}

@patch(as_prop=True)
def lora_targets(self:EncoderSpec):
    "Default LoRA target modules for this family"
    t = FAMILIES.get(self.family, (None, None))[1]
    if t is None: raise ValueError(f"no default LoRA targets for {self.family!r}; pass a peft.LoraConfig")
    return t

@patch
def builder(self:EncoderSpec, template=None, tfms=None) -> RowBuilder:
    "A `RowBuilder` on this encoder's tokenizer (or processor), with its side streams"
    return RowBuilder(self.processor or self.tokenizer, template or self.template, tfms, streams=list(self.streams.values()))

@patch
def to_json(self:EncoderSpec) -> dict:
    "The encoder reference written to `sysone.json`"
    return {"id": self.id, "revision": self.revision, "source": self.source, "hidden_size": self.width,
            "family": self.family, "modality": self.modality}

In [ ]:
spec = EncoderSpec.from_pretrained(FIX)
spec

EncoderSpec(fixtures/tiny-modernbert: modernbert, width 64, context 1024, text, template laya)

In [ ]:
test_eq((spec.width, spec.context, spec.family, spec.modality), (64, 1024, "modernbert", "text"))
test_eq(spec.template.name, "laya")
test_eq(spec.specials["mask"], "[MASK]")
test_eq(spec.lora_targets, r".*\.attn\.(Wqkv|Wo)")

`load_encoder` loads the weights; the encoder is a plain `AutoModel`, returning `last_hidden_state`. It loads in float32 by default, whatever dtype the checkpoint was saved in (NeoMME's is bf16), so the features a head trains on are the features it is served; mixed precision is the Trainer's business. Sources that load differently set `loader`:

In [ ]:
#| export
@patch
def load_encoder(self:EncoderSpec, dtype=torch.float32, **kw):
    "The encoder model, returning one hidden state per position, in float32 unless told otherwise ('auto': the checkpoint's)"
    if self.loader is not None: return self.loader(self, dtype=dtype, **kw)
    return AutoModel.from_pretrained(self.id, revision=self.revision, dtype=dtype, **kw)

In [ ]:
enc = spec.load_encoder()
type(enc).__name__, sum(p.numel() for p in enc.parameters())

('ModernBertModel', 270912)

## DecisionHead

Laya's head: a type embedding (3 × width) added to every position, `layers` pre-norm transformer layers (width / 64 heads, feed-forward 4 × width), and one shared scorer that maps each anchor vector to one logit, LayerNorm → Linear → GELU → Linear. A softmax across a row's anchors gives the option probabilities. `layers=0` keeps only the scorer, the cheapest regime and the one a cache of anchor vectors supports; `scorer="linear"` is a single linear map, the logistic-regression baseline.

$$z_i = w_2^{\top}\,\mathrm{GELU}\!\left(W_1\,\mathrm{LN}(h_{m_i}) + b_1\right) + b_2, \qquad p = \mathrm{softmax}(z / T)$$

The modules are laid out as Laya lays them out, so the parameters map one to one onto a Laya checkpoint (`blocks.i` is Laya's `head.layers.i`). The forward pass, in order:

```{mermaid}
flowchart LR
    H["h (batch, L, width)<br/>the encoder's output"] --> A["+ type_emb(qtype)<br/>broadcast over the row"]
    A --> T["head layers<br/>pre-norm transformer layers,<br/>padding masked"]
    T --> G["gather at marker_pos<br/>(batch, K, width)"]
    G --> S["scorer<br/>LN → Linear → GELU → Linear"]
    S --> Z["logits (batch, K)<br/>padded slots −10⁴"]
```

In [ ]:
#| export
class PairScorer(nn.Module):
    "Scores an option vector against a query vector: an MLP over [o, q, o·q], both projected to `dim` first"
    def __init__(self, width:int, query_width:int, dim:int=256):
        super().__init__()
        self.po, self.pq = nn.Linear(width, dim), nn.Linear(query_width, dim)
        self.mlp = nn.Sequential(nn.LayerNorm(3 * dim), nn.Linear(3 * dim, dim), nn.GELU(), nn.Linear(dim, 1))
    def forward(self, m, q):
        a, b = self.po(m), self.pq(q)[:, None, :].expand(-1, m.shape[1], -1)
        return self.mlp(torch.cat([a, b, a * b], -1))

def make_scorer(kind:str, width:int, query_width:int|None=None, dim:int=256) -> nn.Module:
    "The per-option scorer: `laya` (alias `mlp`), `linear`, `gliner2` (GLiNER2's classifier MLP), or `pair` (against a query vector)"
    if kind in ("laya", "mlp"): return nn.Sequential(nn.LayerNorm(width), nn.Linear(width, width), nn.GELU(), nn.Linear(width, 1))
    if kind == "linear": return nn.Linear(width, 1)
    if kind == "gliner2": return nn.Sequential(nn.Linear(width, 2 * width), nn.ReLU(), nn.Linear(2 * width, 1))
    if kind == "pair":
        if query_width is None: raise ValueError("the pair scorer scores options against a query; give the head query='first' or 'stream:<name>'")
        return PairScorer(width, query_width, dim)
    raise ValueError(f"unknown scorer {kind!r}; use 'laya', 'mlp', 'linear', 'gliner2' or 'pair'")

class DecisionHead(nn.Module):
    "Type embedding, `layers` transformer layers and a shared scorer over the options; optionally a query vector and an act head"
    def __init__(self,
                 width:int, # the encoder's hidden size
                 layers:int=2, # transformer layers over the encoder output; 0 scores the anchors directly
                 scorer:str="laya", # "laya", "mlp", "linear", "gliner2" or "pair"
                 dropout:float=0.1, # dropout in the transformer layers
                 nhead:int|None=None, # attention heads; width // 64 by default
                 init_from:str|None=None, # a Laya checkpoint, a sysone export or a GLiNER2 checkpoint to copy weights from
                 standardize:bool=False, # standardise the encoder's output per dimension before anything else (see `fit_standardizer`)
                 readout:str="anchor", # an option's vector: the one at its "anchor", or the mean over its own tokens ("span")
                 query:str|None=None, # a vector the pair scorer and the act head read: "first" (the row's first position) or "stream:<name>"
                 query_width:int|None=None, # a side stream query's width
                 pair_dim:int=256, # the pair scorer's projection width
                 act:bool=False): # an act head, Laya's: whether to act on the answer or escalate the case
        super().__init__()
        if readout not in ("anchor", "span"): raise ValueError(f"readout must be 'anchor' or 'span', not {readout!r}")
        if query is not None and query != "first" and not str(query).startswith("stream:"):
            raise ValueError(f"query must be None, 'first' or 'stream:<name>', not {query!r}")
        self.width, self.layers, self.scorer_kind, self.dropout, self.standardize = width, layers, scorer, dropout, standardize
        self.readout, self.query, self.pair_dim, self.act = readout, query, pair_dim, act
        self.query_width = None if query is None else width if query == "first" else query_width
        if query is not None and self.query_width is None: raise ValueError(f"give the width of the {query!r} vector (query_width=...)")
        if standardize: self.register_buffer("in_mean", torch.zeros(width)); self.register_buffer("in_std", torch.ones(width))
        if standardize and self.stream_query:
            self.register_buffer("q_mean", torch.zeros(self.query_width)); self.register_buffer("q_std", torch.ones(self.query_width))
        self.nhead = nhead or max(1, width // 64)
        self.type_emb = nn.Embedding(len(QTYPES), width)
        block = nn.TransformerEncoderLayer(width, self.nhead, 4 * width, dropout, batch_first=True, norm_first=True)
        self.blocks = nn.ModuleList([copy.deepcopy(block) for _ in range(layers)])
        self.scorer = make_scorer(scorer, width, self.query_width, pair_dim)
        if act: self.act_head = nn.Sequential(nn.Linear(width + (self.query_width if self.stream_query else 0) + 4, 256), nn.GELU(), nn.Linear(256, 2))
        if init_from: self.load_from(init_from)

    @property
    def stream_query(self) -> str|None:
        "The side stream whose pooled vector is the query, if the query is one"
        return self.query.removeprefix("stream:") if self.query and self.query.startswith("stream:") else None
    @property
    def needs_first(self) -> bool:
        "Whether the head reads the row's first position (an act head, or query='first')"
        return self.act or self.query == "first"

    def config(self) -> dict:
        "The head's shape, as written to `sysone.json`"
        return {"width": self.width, "layers": self.layers, "scorer": self.scorer_kind, "dropout": self.dropout, "nhead": self.nhead,
                "standardize": self.standardize, "readout": self.readout, "query": self.query, "query_width": self.query_width,
                "pair_dim": self.pair_dim, "act": self.act}

    @classmethod
    def from_config(cls, cfg:dict):
        "A head of the shape `config()` describes (weights not included)"
        keys = ("width", "layers", "scorer", "dropout", "nhead", "standardize", "readout", "query", "query_width", "pair_dim", "act")
        return cls(**{k: cfg[k] for k in keys if k in cfg})

In [ ]:
#| export
@patch
def score(self:DecisionHead, m, marker_mask, query=None):
    "Logits from option vectors (batch, K, width), against `query` when the scorer pairs; padded options get -1e4"
    z = self.scorer(m, query) if self.scorer_kind == "pair" else self.scorer(m)
    return z.squeeze(-1).float().masked_fill(~marker_mask.bool(), -1e4)

@patch
def _readout(self:DecisionHead, h, marker_pos, spans=None):
    "One vector per option: the one at its anchor, or the mean over its own tokens (an empty span falls back to the anchor)"
    at = torch.gather(h, 1, marker_pos.clamp(min=0)[:, :, None].expand(-1, -1, h.size(-1)))
    if self.readout == "anchor": return at
    if spans is None: raise ValueError("this head reads option spans (readout='span'); the batch has none")
    cs = torch.cat([h.new_zeros(h.shape[0], 1, h.shape[2]), h.cumsum(1)], 1)
    s, e = spans[..., 0].clamp(0, h.shape[1]), spans[..., 1].clamp(0, h.shape[1])
    pick = lambda i: torch.gather(cs, 1, i[:, :, None].expand(-1, -1, h.size(-1)))
    mean = (pick(e) - pick(s)) / (e - s).clamp(min=1)[:, :, None].to(h.dtype)
    return torch.where((e > s)[:, :, None], mean, at)

@patch
def _decide(self:DecisionHead, m, marker_mask, first=None, query=None):
    "Logits from option vectors, and with an act head `(logits, act_logits)`"
    if self.stream_query and query is None: raise ValueError(f"this head scores options against {self.query!r}; the batch carries no such vector")
    q = first if self.query == "first" else self._standardized_query(query) if self.stream_query else None
    logits = self.score(m, marker_mask, q)
    return (logits, self._act_logits(logits, marker_mask, first, q)) if self.act else logits

@patch
def forward(self:DecisionHead, h, attention_mask, marker_pos, marker_mask, qtype, spans=None, query=None):
    "Logits (batch, K) from the encoder's hidden states (batch, L, width); `(logits, act_logits)` with an act head"
    h = self._standardized(h.to(self.type_emb.weight.dtype)) + self.type_emb(qtype)[:, None, :]
    pad = ~attention_mask.bool()
    for blk in self.blocks: h = blk(h, src_key_padding_mask=pad)
    return self._decide(self._readout(h, marker_pos, spans), marker_mask, h[:, 0], query)

@patch
def forward_anchors(self:DecisionHead, anchors, marker_mask, qtype, query=None, first=None):
    "Logits from cached option vectors (batch, K, width), with the cached first position and query when the head reads them; only for `layers=0`"
    if self.layers: raise ValueError("anchor features only feed a head without layers; cache 'rows' for a head with layers")
    if self.needs_first and first is None: raise ValueError("this head reads the row's first position (an act head, or query='first'); cache it too")
    te = self.type_emb(qtype)
    m = self._standardized(anchors.to(te.dtype)) + te[:, None, :]
    f = self._standardized(first.to(te.dtype)) + te if first is not None else None
    return self._decide(m, marker_mask, f, query)

@patch
def _standardized(self:DecisionHead, x):
    "The encoder's vectors standardised per dimension, when the head standardises"
    return (x - self.in_mean) / self.in_std if self.standardize else x

@patch
def _standardized_query(self:DecisionHead, q):
    "A side stream's query vector, standardised with its own statistics when the head standardises"
    q = q.to(self.type_emb.weight.dtype)
    return (q - self.q_mean) / self.q_std if hasattr(self, "q_mean") else q

@patch(as_prop=True)
def standardizer_fitted(self:DecisionHead) -> bool:
    "Whether the standardisation's statistics have been set (a head without one needs none)"
    unset = lambda m, s: bool((m == 0).all()) and bool((s == 1).all())
    return not self.standardize or not (unset(self.in_mean, self.in_std) or (hasattr(self, "q_mean") and unset(self.q_mean, self.q_std)))

@patch
def fit_standardizer(self:DecisionHead, vectors, query=None):
    "Set the per-dimension mean and standard deviation from encoder vectors (..., width), and a stream query's from `query` (n, query_width)"
    if not self.standardize: raise ValueError("this head does not standardise; build it with DecisionHead(..., standardize=True)")
    v = torch.as_tensor(vectors).float().reshape(-1, self.width)
    with torch.no_grad():
        self.in_mean.copy_(v.mean(0)); self.in_std.copy_(v.std(0).clamp_min(1e-6))
        if hasattr(self, "q_mean"):
            if query is None: raise ValueError(f"this head standardises its {self.query!r} query too; pass its vectors (query=...)")
            qv = torch.as_tensor(query).float().reshape(-1, self.query_width)
            self.q_mean.copy_(qv.mean(0)); self.q_std.copy_(qv.std(0).clamp_min(1e-6))
    return self

In [ ]:
torch.manual_seed(0)
head = DecisionHead(64, layers=2)
h = torch.randn(3, 10, 64)
att = torch.ones(3, 10, dtype=torch.long); att[2, 7:] = 0
mpos = torch.tensor([[1, 4, 6], [2, 5, 0], [1, 3, 5]]); mmask = torch.tensor([[1, 1, 1], [1, 1, 0], [1, 1, 1]]).bool()
qt = torch.tensor([0, 2, 1])
logits = head(h, att, mpos, mmask, qt)
test_eq(logits.shape, (3, 3))
test_eq(logits[1, 2].item(), -1e4)
head

DecisionHead(
  (type_emb): Embedding(3, 64)
  (blocks): ModuleList(
    (0-1): 2 x TransformerEncoderLayer(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (linear1): Linear(in_features=64, out_features=256, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (linear2): Linear(in_features=256, out_features=64, bias=True)
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout1): Dropout(p=0.1, inplace=False)
      (dropout2): Dropout(p=0.1, inplace=False)
    )
  )
  (scorer): Sequential(
    (0): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
    (1): Linear(in_features=64, out_features=64, bias=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)

The three tensor operations of that pass, drawn from tensors the head's code produces (the drawings are in [figures](90_figures.ipynb#the-type-embedding-broadcast-over-a-row)). First, the type embedding: one row of `type_emb.weight` per example, broadcast along the sequence without a copy, so the same anchors read differently in a score row than in a choice row:

![The type embedding picked by qtype and broadcast over the row](images/type_embedding.svg)

Then the gather: one vector per option, the one at its anchor, picked by `torch.gather` along the sequence axis. A padded option slot points at position 0 and gathers a meaningless vector, which the mask hides in the next step:

![torch.gather at marker_pos, from (batch, L, width) to (batch, K, width)](images/anchor_gather.svg)

Last, the scorer turns each anchor vector into a logit, the padded slots are set to $-10^4$, and a softmax across the options, applied by the loss or the `Decider`, gives them exactly zero probability:

![Logits with padded slots at -10^4 and their softmax](images/masked_softmax.svg)

With no layers, scoring cached anchor vectors gives exactly what gathering from the hidden states gives, because the type embedding is the same for every position of a row:

In [ ]:
h0 = DecisionHead(64, layers=0).eval()
full = h0(h, att, mpos, mmask, qt)
anchors = torch.gather(h, 1, mpos[:, :, None].expand(-1, -1, 64))
test_close(h0.forward_anchors(anchors, mmask, qt), full)
test_fail(lambda: head.forward_anchors(anchors, mmask, qt), contains="without layers")

### Readouts and queries

Two further choices shape how the head reads a row.

**The readout** is the vector an option is scored from. Laya's is the anchor's (`readout="anchor"`): the `[MASK]` placed before the option, a position a fine-tuned encoder learns to fill with what the option means in context. A frozen encoder was never taught that, and its anchor vector says more about what word could stand there than about the option after it. `readout="span"` reads the option instead, as the mean of the vectors over its own tokens (the row's `spans`, recorded by the builder). An empty span falls back to the anchor.

**The query** is a vector the options are scored against. `scorer="pair"` scores option $o$ against query $q$ with an MLP over $[W_o o;\ W_q q;\ W_o o \odot W_q q]$, both projected to `pair_dim` first. With `query="first"` the query is the row's first position, `[CLS]`, which has read the question and the whole state. With `query="stream:<name>"` it is the pooled vector of a side stream: the protein, for an encoder whose text stream never sees the protein (see [side streams](#side-streams-two-tower-encoders) below). That is late fusion, and the head is where the two streams meet.

```{mermaid}
flowchart LR
    H["h (batch, L, width)"] --> R{"readout"}
    R -- anchor --> A["h at each anchor"]
    R -- span --> S["mean of h over<br/>each option's tokens"]
    Q["query: h[:, 0], or a side<br/>stream's pooled vector"] --> P["pair scorer<br/>MLP[W_o o; W_q q; W_o o ⊙ W_q q]"]
    A --> P
    S --> P
    P --> Z["logits (batch, K)"]
```

In [ ]:
torch.manual_seed(0)
hs_ = torch.randn(2, 8, 64); att2 = torch.ones(2, 8, dtype=torch.long)
mp2 = torch.tensor([[1, 4], [2, 5]]); mm2 = torch.ones(2, 2, dtype=torch.bool); sp2 = torch.tensor([[[2, 4], [5, 8]], [[3, 5], [6, 6]]])
span_head = DecisionHead(64, layers=0, readout="span", scorer="linear").eval()
got = span_head._readout(hs_, mp2, sp2)
test_close(got[0, 0], hs_[0, 2:4].mean(0))
test_close(got[1, 1], hs_[1, 5])                      # an empty span (6, 6) falls back to its anchor at 5
test_fail(lambda: span_head(hs_, att2, mp2, mm2, torch.zeros(2, dtype=torch.long)), contains="option spans")
pair = DecisionHead(64, layers=0, scorer="pair", query="stream:protein", query_width=32).eval()
z = pair(hs_, att2, mp2, mm2, torch.zeros(2, dtype=torch.long), query=torch.randn(2, 32))
test_eq(z.shape, (2, 2))
test_fail(lambda: pair(hs_, att2, mp2, mm2, torch.zeros(2, dtype=torch.long)), contains="no such vector")
test_fail(lambda: DecisionHead(64, scorer="pair"), contains="give the head query")
test_eq(set(DecisionHead.from_config(pair.config()).state_dict()), set(pair.state_dict()))

### Acting or escalating

An answer is only useful if it can be trusted, and a decision model should say when it can't be. Laya's act head does this. It is a small MLP over the row's first position (`[CLS]`, after the head's layers) and four numbers read off the option probabilities: the top probability, its margin over the second, the entropy normalised by $\log k$, and $k/255$. Its two outputs are *act* (class 0) and *escalate*, and `act_probability` is the softmax's first entry, Laya's convention. `act=True` adds it; with a side stream query, the query vector joins the first position, so the act head sees the protein too. The learner trains it on whether acting would have been right: the gold is among the options and the top option is the gold (see [losses](04_losses.ipynb#the-act-head)). It is not the only way to decide when to act: `calibrate(act_score="confidence")` thresholds the model's own calibrated top probability instead, and on the [protein tutorial](tutorials/protein_decisions.ipynb#acting-or-escalating)'s questions that rule acted on more of them at the same target accuracy.

```{mermaid}
flowchart LR
    Z["logits (batch, K)"] --> F["softmax (detached): top-1, margin,<br/>entropy / log k, k / 255"]
    C["h[:, 0], and a stream query"] --> M["MLP (width + 4 → 256 → 2)"]
    F --> M
    M --> A["p(act) = softmax[0]"]
```

In [ ]:
#| export
@patch
def _act_logits(self:DecisionHead, logits, marker_mask, first, q=None):
    "Laya's act head: the first position (and a stream query) with the top probability, the top-1 margin, the normalised entropy and k/255"
    p = torch.softmax(logits.detach(), -1)
    k = marker_mask.sum(-1).clamp(min=2).float()
    ent = -(p * torch.log(p.clamp_min(1e-9))).sum(-1) / torch.log(k)
    top2 = p.topk(2, -1).values if p.size(-1) >= 2 else torch.cat([p, torch.zeros_like(p)], -1)
    feats = torch.stack([top2[:, 0], top2[:, 0] - top2[:, 1], ent, k / 255.0], -1)
    pooled = first.float() if not self.stream_query else torch.cat([first.float(), q.float()], -1)
    return self.act_head(torch.cat([pooled, feats], -1).to(self.act_head[0].weight.dtype)).float()

def act_probability(act_logits):
    "p(act) from act logits: the first class of a softmax over two, Laya's convention"
    return torch.softmax(torch.as_tensor(act_logits).float(), -1)[..., 0]

In [ ]:
ha = DecisionHead(64, layers=2, act=True).eval()
logits_a, act_a = ha(h, att, mpos, mmask, qt)
test_eq((logits_a.shape, act_a.shape, tuple(ha.act_head[0].weight.shape)), ((3, 3), (3, 2), (256, 68)))   # Laya's shape: width + 4
test_eq(bool(((act_probability(act_a) >= 0) & (act_probability(act_a) <= 1)).all()), True)
hq = DecisionHead(64, layers=0, scorer="pair", query="stream:protein", query_width=32, act=True)
test_eq(tuple(hq.act_head[0].weight.shape), (256, 64 + 32 + 4))
test_fail(lambda: DecisionHead(64, layers=0, act=True).forward_anchors(anchors, mmask, qt), contains="first position")

### Standardising the encoder's output

`standardize=True` subtracts a per-dimension mean from the encoder's vectors and divides by a per-dimension standard deviation before the type embedding, with statistics set once by `fit_standardizer` (the learner does it from the training rows before the first fit). It is an affine map of each vector on its own, so applying it before the gather or to the gathered anchors is the same, and the cached and the live paths agree. The statistics are buffers: they are saved with the head and restored with it, and a warm start from a checkpoint without them leaves them to be fitted.

A side stream's query has statistics of its own (`q_mean`, `q_std`), fitted from the query vectors beside the text's. It exists because of NeoMME, whose vectors have one channel that carries about 97% of every vector's norm (see the [multimodal](20_neomme.ipynb) notebook): a LayerNorm normalises each vector by its own spread, so after it the dominant channel still dwarfs the rest, every anchor looks alike, and a head trained by gradient descent barely moves. Standardising per dimension, across vectors, puts every channel on the same scale first. Off by default, it changes nothing for a Laya-shaped head, and a head that standardises is never written in Laya's format, whose runtime would not apply it.

In [ ]:
torch.manual_seed(0)
hs = DecisionHead(64, layers=0, standardize=True).eval()
test_eq(hs.standardizer_fitted, False)
vecs = torch.randn(200, 64) * 0.1; vecs[:, 5] += 50.0              # one channel carries nearly all of every vector's norm
hs.fit_standardizer(vecs)
test_eq(hs.standardizer_fitted, True)
test_close(((vecs - hs.in_mean) / hs.in_std).mean(0), torch.zeros(64), eps=1e-4)
test_close(hs.forward_anchors(anchors, mmask, qt), hs(h, att, mpos, mmask, qt))      # the same through both paths
test_eq({"in_mean", "in_std"} <= set(hs.state_dict()), True)
test_eq(set(DecisionHead(64, layers=0).state_dict()) & {"in_mean", "in_std"}, set())  # off: no extra state
test_fail(lambda: DecisionHead(64).fit_standardizer(vecs), contains="does not standardise")
hqs = DecisionHead(64, layers=0, scorer="pair", query="stream:protein", query_width=32, standardize=True)
test_fail(lambda: hqs.fit_standardizer(vecs), contains="query too")
hqs.fit_standardizer(vecs, query=torch.randn(50, 32) * 3 + 1)
test_eq(hqs.standardizer_fitted, True)

### Laya's parameter names

`laya_names` maps the head's parameters onto Laya's (`blocks.i.` ↔ `head.layers.i.`); the scorer and type embedding already share Laya's names. A Laya checkpoint's head loads into a `DecisionHead`, and a `DecisionHead` exports into a Laya checkpoint.

In [ ]:
#| export
def to_laya_names(sd:dict) -> dict:
    "Head state-dict keys renamed to Laya's"
    return {re.sub(r"^blocks\.", "head.layers.", k): v for k, v in sd.items()}

def from_laya_names(sd:dict) -> dict:
    "Laya head keys renamed to `DecisionHead`'s, the act head's included (other keys dropped)"
    out = {}
    for k, v in sd.items():
        if k.startswith("head.layers."): out["blocks." + k[len("head.layers."):]] = v
        elif k.startswith(("type_emb.", "scorer.", "act_head.")): out[k] = v
    return out

In [ ]:
test_eq(sorted(from_laya_names(to_laya_names(head.state_dict()))), sorted(head.state_dict()))
[k for k in to_laya_names(head.state_dict()) if not k.startswith("head.layers.1")][:12]

['type_emb.weight',
 'head.layers.0.self_attn.in_proj_weight',
 'head.layers.0.self_attn.in_proj_bias',
 'head.layers.0.self_attn.out_proj.weight',
 'head.layers.0.self_attn.out_proj.bias',
 'head.layers.0.linear1.weight',
 'head.layers.0.linear1.bias',
 'head.layers.0.linear2.weight',
 'head.layers.0.linear2.bias',
 'head.layers.0.norm1.weight',
 'head.layers.0.norm1.bias',
 'head.layers.0.norm2.weight']

### Reading tensors without downloading the file

A warm start from Laya's head needs 25M parameters out of an 842 MB checkpoint. A safetensors file starts with a JSON header giving every tensor's dtype, shape and byte range, so `read_tensors` reads the header and then only the byte ranges it needs, with HTTP range requests, unless the file is already local or cached:

In [ ]:
#| export
_ST_DTYPES = {"F32": np.float32, "F16": np.float16, "BF16": None, "I64": np.int64, "I32": np.int32, "U8": np.uint8, "BOOL": np.bool_}

def _to_tensor(buf, dtype, shape):
    if dtype == "BF16":
        a = np.frombuffer(buf, dtype=np.uint16).astype(np.uint32) << 16
        return torch.from_numpy(a.view(np.float32).reshape(shape).copy()).to(torch.bfloat16)
    return torch.from_numpy(np.frombuffer(buf, dtype=_ST_DTYPES[dtype]).reshape(shape).copy())

def _local_file(repo, filename, revision):
    p = Path(repo) / filename
    if p.exists(): return p
    if Path(repo).exists(): return None
    from huggingface_hub import try_to_load_from_cache
    c = try_to_load_from_cache(repo, filename, revision=revision)
    return Path(c) if isinstance(c, str) else None

def read_tensors(repo:str, filename:str="model.safetensors", select=None, revision:str|None=None) -> dict:
    "Tensors of a safetensors file (local, cached, or on the Hub via range requests) whose names pass `select`"
    select = select or (lambda k: True)
    local = _local_file(repo, filename, revision)
    if local is not None:
        from safetensors import safe_open
        with safe_open(str(local), "pt") as f: return {k: f.get_tensor(k) for k in f.keys() if select(k)}
    from huggingface_hub import hf_hub_url
    from huggingface_hub.utils import build_hf_headers
    url, headers = hf_hub_url(repo, filename, revision=revision), build_hf_headers()
    def get(a, b):
        req = urllib.request.Request(url, headers={**headers, "Range": f"bytes={a}-{b}"})
        with urllib.request.urlopen(req) as r: return r.read()
    n = struct.unpack("<Q", get(0, 7))[0]
    header = json.loads(get(8, 8 + n - 1)); header.pop("__metadata__", None)
    out = {}
    for k, v in header.items():
        if not select(k): continue
        a, b = v["data_offsets"]
        out[k] = _to_tensor(get(8 + n + a, 8 + n + b - 1), v["dtype"], v["shape"]) if b > a else torch.empty(v["shape"])
    return out

In [ ]:
#| eval: false
laya_head = read_tensors("convaiinnovations/laya", select=lambda k: not k.startswith("encoder."))
{k: tuple(v.shape) for k, v in laya_head.items() if not k.startswith("head.layers.1")}

```
{'temperature': (3,), 'act_head.0.bias': (256,), 'act_head.0.weight': (256, 1028), 'act_head.2.bias': (2,),
 'act_head.2.weight': (2, 256), 'head.layers.0.linear1.bias': (4096,), ..., 'scorer.0.bias': (1024,),
 'scorer.0.weight': (1024,), 'scorer.1.bias': (1024,), 'scorer.1.weight': (1024, 1024), 'scorer.3.bias': (1,),
 'scorer.3.weight': (1, 1024), 'type_emb.weight': (3, 1024)}
```

`load_from` fills the head from a checkpoint. `checkpoint(src)` recognises what a folder or a Hub repo holds (optionally a `subfolder` of it) and returns it typed: a `SysoneExport` (`head.safetensors`), a `LayaCheckpoint` (`rl_agent_config.json`), or a kind another module added to `CHECKPOINTS` (a `GlinerCheckpoint`, in the GLiNER encoder notebook). `warm_start(head, checkpoint)` then dispatches on the checkpoint's class (a [generic function](00_core.ipynb#generic-functions-and-multiple-dispatch)). Its default method reads `head_weights(checkpoint)` and checks every name and shape against the head, and a kind with other needs brings its own method. Shapes must match: Laya's head is 1,024 wide, so the warm start is meaningful for ModernBERT-large and nothing else.

In [ ]:
#| export
class Checkpoint:
    "A head's source, a folder or a Hub repo (optionally a `subfolder` of it): its subclass says what its files hold"
    def __init__(self, src:str, subfolder:str|None=None): self.src, self.subfolder = str(src), subfolder
    def __repr__(self): return f"{type(self).__name__}({self.src!r}" + (f", {self.subfolder!r})" if self.subfolder else ")")

class SysoneExport(Checkpoint): "A sysone export: the head's weights in `head.safetensors`"
class LayaCheckpoint(Checkpoint): "A Laya checkpoint: `rl_agent_config.json`, and the head beside the encoder in `model.safetensors`"

def split_repo(src:str):
    "A Hub id with an optional subfolder: 'owner/repo/sub' -> ('owner/repo', 'sub'); local paths pass through"
    src = str(src)
    if Path(src).exists() or src.startswith(("/", ".", "~")) or src.count("/") < 2: return src, None
    parts = src.split("/")
    return "/".join(parts[:2]), "/".join(parts[2:])

def _is_laya(src, subfolder=None):
    p = Path(src) / (subfolder or "")
    if p.exists(): return (p / "rl_agent_config.json").exists()
    try:
        from huggingface_hub import file_exists
        return file_exists(src, f"{subfolder + '/' if subfolder else ''}rl_agent_config.json")
    except Exception: return False

def _is_export(src, subfolder=None): return (Path(src) / (subfolder or "") / "head.safetensors").exists()

CHECKPOINTS = [(_is_export, SysoneExport), (_is_laya, LayaCheckpoint)]    # (detect(src, subfolder) -> bool, kind), tried in order

def checkpoint(src, subfolder:str|None=None) -> Checkpoint:
    "The checkpoint at `src`, typed by the first kind in `CHECKPOINTS` whose files it holds"
    if isinstance(src, Checkpoint): return src
    if subfolder is None: src, subfolder = split_repo(src)
    for detect, kind in CHECKPOINTS:
        if detect(src, subfolder): return kind(src, subfolder)
    raise ValueError(f"{src!r} is not a Laya checkpoint, a sysone export or a known head source")

@dispatch
def head_weights(ckpt:SysoneExport) -> dict:
    "A sysone export's head weights"
    return read_tensors(str(Path(ckpt.src) / (ckpt.subfolder or "")), "head.safetensors")

@dispatch
def head_weights(ckpt:LayaCheckpoint) -> dict:
    "A Laya checkpoint's head weights, under sysone's names"
    fn = f"{ckpt.subfolder}/model.safetensors" if ckpt.subfolder else "model.safetensors"
    return from_laya_names(read_tensors(ckpt.src, fn, select=lambda k: not k.startswith("encoder.")))

@dispatch
def warm_start(head:DecisionHead, ckpt:Checkpoint, *, strict:bool=True):
    "Copy a checkpoint's head weights into `head`, every name and shape checked: dispatched on the checkpoint's kind"
    sd, own, src = head_weights(ckpt), head.state_dict(), ckpt.src
    if not head.act: sd = {k: v for k, v in sd.items() if not k.startswith("act_head.")}     # a head without one ignores it
    bad = [f"{k}: {tuple(v.shape)} vs {tuple(own[k].shape)}" for k, v in sd.items() if k in own and v.shape != own[k].shape]
    if bad: raise ValueError(f"head shapes do not match {src!r} (width {head.width}, {head.layers} layers): {bad[:3]}")
    missing = [k for k in own if k not in sd and k not in ("in_mean", "in_std")]    # statistics of an encoder, not weights
    extra = [k for k in sd if k not in own]
    if missing and strict: raise ValueError(f"{src!r} lacks head parameters {missing[:3]}")
    if extra and strict: raise ValueError(f"{src!r} has head parameters this head lacks ({extra[:3]}): match its layers, or strict=False")
    head.load_state_dict({k: v.to(own[k].dtype) for k, v in sd.items() if k in own}, strict=False)
    return head

@patch
def load_from(self:DecisionHead, src, subfolder:str|None=None, strict:bool=True):
    "Copy weights from a Laya checkpoint, a sysone export or any kind in `CHECKPOINTS`"
    return warm_start(self, checkpoint(src, subfolder), strict=strict)

## EncoderDecisionModel

`EncoderDecisionModel(encoder, head)` runs the encoder and the head: `forward(input_ids, attention_mask, marker_pos, marker_mask, qtype, **encoder_kwargs)` returns logits of shape (batch, K), or `(logits, act_logits)` with an act head. A head whose query is a side stream gets it from the encoder's output (`encode_full`), and a span readout gets the batch's `spans`. Everything the encoder needs beyond ids rides in `encoder_kwargs` (`pixel_values`, `position_ids`, `image_grid_hw`), so the head, the gather and the losses never see the modality. Cached features skip the encoder: `anchors` (batch, K, width) feed a head without layers, `hidden` (batch, L, width) feed any head. With a frozen encoder (`frozen=True`) the encoder runs under `no_grad` and stays in eval mode. An encoder that wants its inputs reshaped, NeoMME's two-axis positions or ModernVBERT's tiles, gets them through `encoder_inputs`, a [generic function](00_core.ipynb#generic-functions-and-multiple-dispatch) dispatched on the encoder's config class, to which its adapter adds a method.

In [ ]:
#| export
@dispatch
def encoder_inputs(config:object, inputs:dict) -> dict:
    "The keyword arguments an encoder is called with, reshaped where its adapter says so: dispatched on the encoder config's class"
    if load_adapter(config): return encoder_inputs(config, inputs)
    return inputs

class EncoderDecisionModel(nn.Module):
    "An encoder and a `DecisionHead`: logits over each row's options"
    def __init__(self, encoder:nn.Module, head:DecisionHead, spec:EncoderSpec|None=None):
        super().__init__()
        self.encoder, self.head, self.spec = encoder, head, spec
        self.frozen, self.regime = False, "full"

    def encode_full(self, input_ids, attention_mask, **encoder_kwargs):
        "The encoder's output: its `last_hidden_state`, and a stream encoder's `stream_vectors`"
        encoder_kwargs = encoder_inputs(getattr(self.encoder, "config", None), encoder_kwargs)
        with torch.no_grad() if self.frozen else nullcontext():
            return self.encoder(input_ids=input_ids, attention_mask=attention_mask, **encoder_kwargs)

    def encode(self, input_ids, attention_mask, **encoder_kwargs):
        "The encoder's last hidden state"
        return self.encode_full(input_ids, attention_mask, **encoder_kwargs).last_hidden_state

    def stream_query(self, out):
        "The head's query vector from an encoder output, when the query is a side stream's"
        name = self.head.stream_query
        if name is None: return None
        vecs = getattr(out, "stream_vectors", None) or {}
        if name not in vecs: raise ValueError(f"the head's query is stream {name!r}, which the encoder did not return: is it a StreamEncoder with that stream, and does the batch carry it?")
        return vecs[name]

    def forward(self, input_ids=None, attention_mask=None, marker_pos=None, marker_mask=None, qtype=None,
                anchors=None, hidden=None, query=None, first=None, spans=None, answerable=None, **encoder_kwargs):
        if anchors is not None: return self.head.forward_anchors(anchors, marker_mask, qtype, query=query, first=first)
        if hidden is None:
            out = self.encode_full(input_ids, attention_mask, **encoder_kwargs)
            hidden, query = out.last_hidden_state, query if query is not None else self.stream_query(out)
        return self.head(hidden, attention_mask, marker_pos, marker_mask, qtype, spans=spans, query=query)

    def train(self, mode:bool=True):
        super().train(mode)
        if self.frozen: self.encoder.eval()
        return self

In [ ]:
#| export
@patch
def gradient_checkpointing_enable(self:EncoderDecisionModel, gradient_checkpointing_kwargs=None):
    "Checkpoint the encoder's layers (what `TrainingArguments(gradient_checkpointing=True)` calls)"
    kw = {"use_reentrant": False} | (gradient_checkpointing_kwargs or {})
    self.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs=kw)

@patch
def gradient_checkpointing_disable(self:EncoderDecisionModel): self.encoder.gradient_checkpointing_disable()

@patch(as_prop=True)
def device(self:EncoderDecisionModel): return self.head.type_emb.weight.device

@patch
def n_params(self:EncoderDecisionModel) -> dict:
    "Total and trainable parameter counts, for the encoder and the head"
    c = lambda m, t: sum(p.numel() for p in m.parameters() if p.requires_grad or not t)
    return {"encoder": c(self.encoder, False), "head": c(self.head, False),
            "trainable": c(self.encoder, True) * (not self.frozen) + c(self.head, True)}

In [ ]:
from transformers import AutoTokenizer

In [ ]:
tok = AutoTokenizer.from_pretrained(FIX)
recs = [json.loads(l) for l in Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()]
rows = spec.builder().build_record(recs[0])
batch = collate(rows, tok.pad_token_id)
model = EncoderDecisionModel(enc, DecisionHead(spec.width), spec)
out = model(**{k: v for k, v in batch.items() if k not in ("target", "label")})
test_eq(out.shape, (5, 4))
test_eq("spans" in batch and "answerable" in batch, True)          # both ride in the batch; the model reads spans, ignores answerable
span_model = EncoderDecisionModel(enc, DecisionHead(spec.width, layers=0, readout="span"), spec)
test_eq(span_model(**{k: v for k, v in batch.items() if k not in ("target", "label")}).shape, (5, 4))
test_eq([r.meta["qid"] for r in rows][1], "needs_review")
test_eq((out[1] == -1e4).sum().item(), 2)       # the noul row has two options
out.softmax(-1)[1]

tensor([0.5045, 0.4955, 0.0000, 0.0000], grad_fn=<SelectBackward0>)

## Side streams: two-tower encoders

Some encoders read two inputs with two towers: ProtST pairs ESM (a protein language model) with PubMedBERT (a biomedical text model), and CLIP-style models pair an image tower with a text tower. `StreamEncoder(text, sides)` holds a text encoder and one `SideEncoder` per side stream. The text encoder reads the row as always, and each side encoder reads its stream's tokens (the `{name}_input_ids` a `TokenStream` put in the batch) and pools them to one vector, the mean over its non-special tokens, optionally projected. The output carries the text's `last_hidden_state` for the anchors and each stream's pooled vector in `stream_vectors`, which a head reads as its query (`query="stream:<name>"`).

Here the two towers do not talk (late fusion). The text stream never sees the protein, so all the joining happens in the head, which is cheap: both towers can stay frozen and cached. [Cross-attention](#cross-attention-from-a-side-stream) below lets the text attend to the stream inside the encoder instead.

A side stream is shared by every question about the same state, so one batch often holds the same protein several times. The side encoder runs once per distinct input in a batch and hands the result back to every row that holds it.

```{mermaid}
flowchart LR
    B["batch: input_ids,<br/>protein_input_ids"] --> T["text encoder<br/>(the row: question, anchors, options)"]
    B --> D["distinct proteins<br/>of the batch"]
    D --> S["side encoder<br/>(ESM), pooled"]
    T --> H["last_hidden_state"]
    S --> V["stream_vectors['protein']<br/>(back to every row)"]
    H --> HD["head: anchors ⨉ query"]
    V --> HD
```

In [ ]:
#| export
@dataclass
class StreamOutput:
    "A stream encoder's output: the text's hidden states, and each side stream's pooled vector"
    last_hidden_state: torch.Tensor
    stream_vectors: dict = field(default_factory=dict)

def distinct_rows(ids, mask) -> tuple:
    "The first index of each distinct (ids, mask) row of a batch, and each row's distinct index"
    seen, first, inverse = {}, [], []
    for i, key in enumerate(map(tuple, torch.cat([ids, mask.to(ids.dtype)], -1).tolist())):
        j = seen.setdefault(key, len(first))
        if j == len(first): first.append(i)
        inverse.append(j)
    return first, inverse

class ProjMLP(nn.Sequential):
    "Linear → ReLU → Linear: the projection head of a CLIP-style tower"
    def __init__(self, dims):
        super().__init__(nn.Linear(dims[0], dims[1]), nn.ReLU(), nn.Linear(dims[1], dims[2])); self.dims = list(dims)

class SideEncoder(nn.Module):
    "A side stream's encoder: a transformers model over its tokens, pooled to one vector per input (the mean over its non-special tokens, optionally projected), run once per distinct input of a batch"
    def __init__(self, model:nn.Module, special_ids=(), proj:ProjMLP|None=None):
        super().__init__()
        self.model, self.proj, self.special_ids = model, proj, [int(i) for i in special_ids]
        self.memo = None                            # {input: pooled vector} while `stream_memo` holds one
        self.dedupe = True                          # encode each distinct input once; an attribution turns it off (its batch is one row, repeated)
    @property
    def width(self) -> int: return self.model.config.hidden_size
    @property
    def out_width(self) -> int: return self.proj.dims[-1] if self.proj is not None else self.width

    def _run(self, ids, mask):
        h = self.model(input_ids=ids, attention_mask=mask).last_hidden_state
        keep = mask.bool()
        for s in self.special_ids: keep = keep & (ids != s)
        pooled = (h * keep[..., None].to(h.dtype)).sum(1) / keep.sum(1, keepdim=True).clamp(min=1).to(h.dtype)
        return h, (self.proj(pooled) if self.proj is not None else pooled)

    def forward(self, ids, mask, states:bool=False):
        "(hidden states (batch, P, width) if asked, else None; pooled vectors (batch, out_width))"
        first, inverse = distinct_rows(ids, mask) if self.dedupe else (list(range(len(ids))),) * 2
        inv = torch.tensor(inverse, device=ids.device)
        if self.memo is None or states or torch.is_grad_enabled():
            h, pooled = self._run(ids[first], mask[first])
            return (h[inv] if states else None), pooled[inv]
        keys = [ids[i][mask[i].bool()].cpu().numpy().tobytes() for i in first]
        todo = [j for j, k in enumerate(keys) if k not in self.memo]
        if todo:
            rows = [first[j] for j in todo]
            self.memo.update(zip([keys[j] for j in todo], self._run(ids[rows], mask[rows])[1]))
        return None, torch.stack([self.memo[k] for k in keys])[inv]

@contextmanager
def stream_memo(module:nn.Module):
    "While the block runs, every side encoder in `module` encodes each distinct input once, however the rows are batched (no-grad passes)"
    sides = [m for m in module.modules() if isinstance(m, SideEncoder)]
    for s in sides: s.memo = {}
    try: yield module
    finally:
        for s in sides: s.memo = None

class StreamEncoder(nn.Module):
    "A text encoder beside side-stream encoders: with `late` fusion they run apart and the head joins them; with `xattn` the text attends to a stream"
    base_model_prefix = ""                          # peft reads it off a model with a config: the towers sit at the top level
    def __init__(self, text:nn.Module, sides:dict):
        super().__init__()
        self.text, self.sides = text, nn.ModuleDict(sides)
        self.proj = nn.ModuleDict()                 # stream -> text width, for cross-attention
        self.config = text.config                   # the text encoder's: what the width, the family and the hooks read
        self.fusion, self.xattn, self._memory = "late", None, {}

    def forward(self, input_ids=None, attention_mask=None, **kw):
        vectors, memory = {}, {}
        for name, side in self.sides.items():
            ids, mask = kw.pop(f"{name}_input_ids", None), kw.pop(f"{name}_attention_mask", None)
            if ids is None: continue
            need = name in self.proj
            states, vectors[name] = side(ids, mask, states=need)
            if need: memory[name] = (self.proj[name](states), mask.bool())
        self._memory = memory
        try: out = self.text(input_ids=input_ids, attention_mask=attention_mask, **kw)
        finally: self._memory = {}
        return StreamOutput(out.last_hidden_state, vectors)

    def gradient_checkpointing_enable(self, **kw): self.text.gradient_checkpointing_enable(**kw)
    def gradient_checkpointing_disable(self): self.text.gradient_checkpointing_disable()
    def save_pretrained(self, path, **kw): return save_stream_encoder(self, path)

On the fixture, the text tower is the tiny ModernBERT and the side tower a tiny random ESM built from its config. Five questions about three proteins: the protein tower runs on three rows, and rows with the same protein get the same vector:

In [ ]:
from transformers import EsmConfig, EsmModel

In [ ]:
torch.manual_seed(0)
esm = EsmModel(EsmConfig(vocab_size=33, hidden_size=32, num_hidden_layers=2, num_attention_heads=2, intermediate_size=64,
                         max_position_embeddings=64, pad_token_id=1, mask_token_id=32, position_embedding_type="absolute",
                         token_dropout=False), add_pooling_layer=False).eval()
se = StreamEncoder(spec.load_encoder(), {"protein": SideEncoder(esm, special_ids=(0, 1, 2))}).eval()
EXTRA_COLLATE["protein_input_ids"], EXTRA_COLLATE["protein_attention_mask"] = ("pad_rows", 1), ("pad_rows", 0)
prot = lambda n: {"protein_input_ids": [0] + [4 + (i * 7) % 20 for i in range(n)] + [2], "protein_attention_mask": [1] * (n + 2)}
prows = spec.builder().build_record(recs[0])
for r, n in zip(prows, (10, 10, 6, 6, 14)): r.extras.update(prot(n))
pb = collate(prows, tok.pad_token_id)
pin = {k: v for k, v in pb.items() if k not in ("target", "label")}
enc_in = {k: pin[k] for k in ("input_ids", "attention_mask", "protein_input_ids", "protein_attention_mask")}
seen_rows = []
hook = esm.register_forward_pre_hook(lambda m, a, kw: seen_rows.append(kw["input_ids"].shape[0]), with_kwargs=True)
with torch.no_grad(): sout = se(**enc_in)
hook.remove()
test_eq((sout.last_hidden_state.shape[0], sout.stream_vectors["protein"].shape, seen_rows), (5, (5, 32), [3]))
test_close(sout.stream_vectors["protein"][0], sout.stream_vectors["protein"][1])
pm = EncoderDecisionModel(se, DecisionHead(spec.width, layers=0, scorer="pair", query="stream:protein", query_width=32), spec).eval()
with torch.no_grad(): test_eq(pm(**pin).shape, (5, 4))
test_fail(lambda: EncoderDecisionModel(spec.load_encoder(), pm.head, spec)(**{k: v for k, v in pin.items() if not k.startswith("protein")}),
          contains="did not return")

The distinct-input rule works within a batch. A pass over many rows, such as a feature cache's, batches them by length, so the rows of one protein usually land in different batches and the protein tower would run once per question. `stream_memo(encoder)` makes every side encoder remember the vectors it has computed while the block runs, and so each distinct input runs once per pass. It only applies to no-grad passes, where a vector can be reused, and the feature cache encodes under it. Here, a second pass over the same three proteins, in another order, doesn't run the protein tower at all:

In [ ]:
calls = []
hook = esm.register_forward_pre_hook(lambda m, a, kw: calls.append(kw["input_ids"].shape[0]), with_kwargs=True)
with torch.no_grad(), stream_memo(se):
    first_pass = se(**enc_in).stream_vectors["protein"]
    second_pass = se(**{k: v[[4, 3, 2, 1, 0]] for k, v in enc_in.items()}).stream_vectors["protein"]
hook.remove()
test_eq(calls, [3])
test_close(first_pass, sout.stream_vectors["protein"], eps=1e-6); test_close(second_pass, first_pass[[4, 3, 2, 1, 0]], eps=1e-6)
test_eq(se.sides["protein"].memo, None)

## Regimes

The regime says what trains. `head` trains the head only, with the encoder frozen; with a feature cache the encoder does not even run after one pass. `lora` puts LoRA adapters on the attention projections, `mica` is LoRA with MiCA's initialisation (B from the smallest singular vectors of the weight, frozen; A trains), and `full` trains everything, with discriminative learning rates set by the learner. A `peft` config instance works too. The adapters live in the encoder, which becomes a `PeftModel`; nothing else changes.

| `train=` | encoder | adapters | head | feature cache |
| --- | --- | --- | --- | --- |
| `"head"` | frozen, eval mode, no gradient | none | trains | yes: the encoder runs once |
| `"lora"` | frozen | A and B train | trains | no |
| `"mica"` | frozen | A trains; B frozen at the weight's minor singular vectors | trains | no |
| `"full"` | trains, at a quarter of the head's learning rate | none | trains | no |
| `"xattn"` | frozen, but runs with gradients | the cross-attention blocks and their projection train | trains | no |

In [ ]:
#| export
def lora_config(regime:str="lora", spec:EncoderSpec|None=None, r:int=8, alpha:int=16, dropout:float=0.05, targets=None, **kw):
    "The `peft.LoraConfig` of the `lora` or `mica` regime"
    from peft import LoraConfig
    targets = targets or (spec.lora_targets if spec is not None else None)
    if targets is None: raise ValueError("name the LoRA target modules (targets=...) or pass an EncoderSpec")
    if regime == "mica": kw.setdefault("init_lora_weights", "mica")
    return LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, target_modules=targets, **kw)

def set_regime(model:EncoderDecisionModel, regime="head", **lora_kw) -> EncoderDecisionModel:
    "Set what trains: 'head', 'lora', 'mica', 'full', 'xattn' (a stream encoder's cross-attention), or a peft config"
    from peft import PeftConfig, get_peft_model
    for p in model.head.parameters(): p.requires_grad_(True)
    if isinstance(regime, PeftConfig) or regime in ("lora", "mica"):
        cfg = regime if isinstance(regime, PeftConfig) else lora_config(regime, model.spec, **lora_kw)
        model.encoder = get_peft_model(model.encoder, cfg)
        model.frozen, model.regime = False, regime if isinstance(regime, str) else type(regime).__name__
        # exactly what peft left trainable: MiCA keeps its B matrices frozen, and unfreezing must not thaw them
        model.adapter_params = {n for n, p in model.encoder.named_parameters() if p.requires_grad}
        model.adapter_regime = model.regime
    elif regime == "head":
        for p in model.encoder.parameters(): p.requires_grad_(False)
        model.frozen, model.regime = True, "head"
    elif regime == "full":
        for p in model.encoder.parameters(): p.requires_grad_(True)
        model.frozen, model.regime = False, "full"
    elif regime == "xattn":
        if getattr(model.encoder, "fusion", None) != "xattn": raise ValueError("the xattn regime needs cross-attention: add_cross_attention(model.encoder, stream) first")
        for n, p in model.encoder.named_parameters(): p.requires_grad_(".block." in n or n.startswith("proj."))
        model.frozen, model.regime = False, "xattn"
    else: raise ValueError(f"unknown regime {regime!r}; use 'head', 'lora', 'mica', 'full', 'xattn' or a peft config")
    model.train(model.training)
    return model

def thaw_adapters(model:EncoderDecisionModel) -> EncoderDecisionModel:
    "Make trainable again exactly the adapter parameters `set_regime` made trainable"
    for n, p in model.encoder.named_parameters(): p.requires_grad_(n in model.adapter_params)
    for p in model.head.parameters(): p.requires_grad_(True)
    model.frozen, model.regime = False, model.adapter_regime
    model.train(model.training)
    return model

In [ ]:
def fresh(regime, **kw):
    torch.manual_seed(0)
    return set_regime(EncoderDecisionModel(spec.load_encoder(), DecisionHead(spec.width), spec), regime, **kw)

m_head, m_lora, m_full = fresh("head"), fresh("lora"), fresh("full")
{n: m.n_params() for n, m in [("head", m_head), ("lora", m_lora), ("full", m_full)]}

{'head': {'encoder': 270912, 'head': 104513, 'trainable': 104513},
 'lora': {'encoder': 283200, 'head': 104513, 'trainable': 116801},
 'full': {'encoder': 270912, 'head': 104513, 'trainable': 375425}}

In [ ]:
test_eq(m_head.n_params()["trainable"], m_head.n_params()["head"])
assert m_head.n_params()["head"] < m_lora.n_params()["trainable"] < m_full.n_params()["trainable"]
lora_names = [n for n, p in m_lora.encoder.named_parameters() if p.requires_grad]
assert all("lora_" in n and ".attn." in n for n in lora_names)
m_head.train(); test_eq(m_head.encoder.training, False)

::: {.callout-note title="Finding: BERT's LoRA targets match nothing in ModernBERT"}
The usual LoRA targets, `query`, `key` and `value`, are BERT's names. ModernBERT projects queries, keys and values with one fused `Wqkv`, and it has a `Wo` twice per layer: the attention's output (`attn.Wo`) and the MLP's (`mlp.Wo`). So BERT's list finds nothing and peft refuses the model, and a plain `["Wqkv", "Wo"]` would also put adapters on the MLP; the family table's target is a regular expression that keeps them on the attention:
:::

In [ ]:
test_fail(lambda: set_regime(EncoderDecisionModel(spec.load_encoder(), DecisionHead(spec.width), spec), lora_config(targets=["query", "key", "value"])),
          contains="not found in the base model")
names = [n for n, _ in spec.load_encoder().named_modules()]
[n for n in names if n.startswith("layers.0.") and n.endswith(("Wqkv", "Wo"))], [n for n in names if re.fullmatch(spec.lora_targets, n)][:4]

(['layers.0.attn.Wqkv', 'layers.0.attn.Wo', 'layers.0.mlp.Wo'],
 ['layers.0.attn.Wqkv',
  'layers.0.attn.Wo',
  'layers.1.attn.Wqkv',
  'layers.1.attn.Wo'])

A frozen encoder gets no gradient, a LoRA one only on its adapters:

In [ ]:
inputs = {k: v for k, v in batch.items() if k not in ("target", "label")}
for m in (m_head, m_lora):
    m.train(); m.zero_grad()
    m(**inputs).logsumexp(-1).sum().backward()
test_eq(all(p.grad is None for p in m_head.encoder.parameters()), True)
assert all(p.grad is not None for n, p in m_lora.encoder.named_parameters() if p.requires_grad)

`freeze_to(n)` freezes the embeddings and the first n layers of the encoder, fastai style, for partial fine-tuning; `unfreeze` undoes it. The layers are found as the encoder's longest `ModuleList` of identical blocks, so it works across families:

In [ ]:
#| export
def encoder_layers(encoder:nn.Module) -> nn.ModuleList:
    "The encoder's stack of transformer layers"
    base = encoder.get_base_model() if hasattr(encoder, "get_base_model") else encoder
    best, kind = None, lambda c: type(c.layer) if getattr(c, "is_xattn", False) else type(c)     # a cross-attention wrapper counts as its layer
    for m in base.modules():
        if isinstance(m, nn.ModuleList) and len(m) and len({kind(c) for c in m}) == 1 and (best is None or len(m) > len(best)): best = m
    if best is None: raise ValueError("could not find the encoder's layers")
    return best

def freeze_to(model:EncoderDecisionModel, n:int):
    "Freeze the encoder's embeddings and its first n layers; train the rest (on an adapter model, the rest's adapters)"
    adapters = getattr(model, "adapter_params", None) and hasattr(model.encoder, "peft_config")
    if adapters: thaw_adapters(model)
    else: set_regime(model, "full")
    allowed = {id(p) for nm, p in model.encoder.named_parameters() if nm in model.adapter_params} if adapters else None
    layers = encoder_layers(model.encoder)
    base = model.encoder.get_base_model() if hasattr(model.encoder, "get_base_model") else model.encoder
    layer_params = {id(p) for l in layers for p in l.parameters()}
    for p in base.parameters():
        if id(p) not in layer_params: p.requires_grad_(False)          # embeddings
    for i, l in enumerate(layers):
        for p in l.parameters(): p.requires_grad_(i >= n and (allowed is None or id(p) in allowed))
    if not adapters:
        for name, mod in base.named_children():                         # a final norm after the stack trains
            if "norm" in name.lower():
                for p in mod.parameters(): p.requires_grad_(True)
    model.regime = f"freeze_to({n})"
    return model

In [ ]:
m = freeze_to(fresh("full"), 2)
layers = encoder_layers(m.encoder)
test_eq([all(p.requires_grad for p in l.parameters()) for l in layers], [False, False, True, True])
test_eq(m.encoder.embeddings.tok_embeddings.weight.requires_grad, False)

MiCA trains fewer parameters than LoRA: B is set from the weight's smallest singular vectors and frozen, only A trains. On a square projection that is half of LoRA's count; ModernBERT's fused `Wqkv` is three times as wide as it is tall, so here it is less than half.

In [ ]:
m_mica = fresh("mica")
mica_trainable = sum(p.numel() for p in m_mica.encoder.parameters() if p.requires_grad)
lora_trainable = sum(p.numel() for p in m_lora.encoder.parameters() if p.requires_grad)
assert mica_trainable < lora_trainable
test_eq(all("lora_A" in n for n, p in m_mica.encoder.named_parameters() if p.requires_grad), True)
mica_trainable, lora_trainable

(4096, 12288)

::: {.callout-note title="Finding: unfreezing thawed MiCA's frozen matrices"}
The first `Learner.unfreeze` made every adapter parameter trainable again, and on a MiCA run that includes the B matrices MiCA keeps frozen: the run silently became LoRA with an unusual initialisation. `set_regime` now records exactly what peft left trainable (`adapter_params`), and `thaw_adapters`, `unfreeze` and `freeze_to` restore that set and nothing more.
:::

The regime remembers which adapter parameters it made trainable, so thawing after a freeze, or freezing the lower layers, never unfreezes MiCA's B matrices:

In [ ]:
set_regime(m_mica, "head"); thaw_adapters(m_mica)
test_eq((sum(p.numel() for p in m_mica.encoder.parameters() if p.requires_grad), m_mica.regime), (mica_trainable, "mica"))
freeze_to(m_mica, 2)
test_eq(any(p.requires_grad for n, p in m_mica.encoder.named_parameters() if "lora_B" in n), False)
test_eq(any(p.requires_grad for n, p in m_mica.encoder.named_parameters() if "layers.3." in n and "lora_A" in n), True)

## Cross-attention from a side stream

Late fusion keeps a text encoder that never reads the protein. Design B in the protein note goes further: it keeps a trained text decision model whole (Laya's ModernBERT with its head) and lets its text attend to a frozen protein encoder's residue states, through new cross-attention layers. `add_cross_attention(stream_encoder, stream, every=4)` inserts a gated cross-attention block after every fourth layer of the text encoder. Each block reads the side stream's hidden states, projected to the text's width, and adds $\tanh(g)$ times what it reads. The gate $g$ starts at zero, as in Flamingo, so at the first step the model is exactly the text model it was, and training opens the gates as the new path starts to help. The `xattn` regime trains only the projection, the blocks and the head.

```{mermaid}
flowchart LR
    P["side stream<br/>(frozen ESM states)"] --> W["projection<br/>to the text width"]
    L1["text layer 4k"] --> X["gated cross-attention<br/>h + tanh(g) · attn(LN(h), memory)"]
    W --> X
    X --> L2["text layer 4k + 1"]
```

In [ ]:
#| export
class GatedCrossAttention(nn.Module):
    "Text states attend to a side stream's states; `tanh(gate)` scales what they read, and the gate starts at 0, so the block starts as the identity"
    def __init__(self, width:int, heads:int):
        super().__init__()
        self.norm, self.attn = nn.LayerNorm(width), nn.MultiheadAttention(width, heads, batch_first=True)
        self.gate = nn.Parameter(torch.zeros(1))
    def forward(self, h, memory, memory_mask):
        a, _ = self.attn(self.norm(h), memory, memory, key_padding_mask=~memory_mask, need_weights=False)
        return h + torch.tanh(self.gate).to(h.dtype) * a

class XAttnLayer(nn.Module):
    "An encoder layer followed by a gated cross-attention block over its stream encoder's current memory"
    is_xattn = True
    def __init__(self, layer:nn.Module, block:GatedCrossAttention, owner:nn.Module, stream:str):
        super().__init__()
        self.layer, self.block, self.stream = layer, block, stream
        object.__setattr__(self, "_owner", owner)          # a reference, not a submodule: the owner holds this layer
    def __getattr__(self, name):                            # what the encoder reads off its layers (ModernBERT: attention_type)
        try: return super().__getattr__(name)
        except AttributeError: return getattr(super().__getattr__("layer"), name)
    def forward(self, *args, **kw):
        out = self.layer(*args, **kw)
        mem = self._owner._memory.get(self.stream)
        if mem is None: return out
        h = out[0] if isinstance(out, tuple) else out
        if h.dim() != 3: raise ValueError("cross-attention needs padded (batch, length, width) states; this encoder runs unpadded")
        h = self.block(h, *mem)
        return (h, *out[1:]) if isinstance(out, tuple) else h

def add_cross_attention(encoder:StreamEncoder, stream:str, every:int=4, heads:int|None=None) -> StreamEncoder:
    "Insert a gated cross-attention block over `stream` after every `every`-th layer of the text encoder (in place)"
    if stream not in encoder.sides: raise ValueError(f"no side stream {stream!r}; the encoder has {list(encoder.sides)}")
    if encoder.xattn is not None: raise ValueError("this encoder already has cross-attention")
    layers, width = encoder_layers(encoder.text), encoder.text.config.hidden_size
    heads = heads or max(1, width // 64)
    encoder.proj[stream] = nn.Linear(encoder.sides[stream].width, width)
    for i in range(every - 1, len(layers), every): layers[i] = XAttnLayer(layers[i], GatedCrossAttention(width, heads), encoder, stream)
    encoder.fusion, encoder.xattn = "xattn", {"stream": stream, "every": every, "heads": heads}
    return encoder

On the fixture, cross-attention every second layer of the four-layer text encoder adds two blocks. With the gates at zero, the text encoder's output is exactly what it was. One step of the `xattn` regime moves the gates, trains only the new parts, and changes the output:

In [ ]:
xse = add_cross_attention(copy.deepcopy(se), "protein", every=2).eval()
with torch.no_grad(): test_close(xse(**enc_in).last_hidden_state, sout.last_hidden_state, eps=1e-6)
test_eq(sum(getattr(l, "is_xattn", False) for l in encoder_layers(xse.text)), 2)
xm = set_regime(EncoderDecisionModel(xse, DecisionHead(spec.width, layers=0), spec), "xattn")
test_eq({n.split(".")[0] for n, p in xm.encoder.named_parameters() if p.requires_grad}, {"text", "proj"})
test_eq(all(".block." in n or n.startswith("proj.") for n, p in xm.encoder.named_parameters() if p.requires_grad), True)
opt = torch.optim.SGD([p for p in xm.parameters() if p.requires_grad], lr=1.0)
nn.functional.cross_entropy(xm(**pin), pb["label"]).backward(); opt.step()
gates = [l.block.gate.item() for l in encoder_layers(xse.text) if getattr(l, "is_xattn", False)]
test_eq(all(g != 0 for g in gates), True)
xm.eval()
with torch.no_grad(): test_ne(xse(**enc_in).last_hidden_state, sout.last_hidden_state)
gates

[1.479590764574823e-06, 7.796261343173683e-07]

### Saving a stream encoder

A stream encoder saves as transformers folders, the text encoder (without its cross-attention blocks, so any transformers version reads it) and each side model, plus a small file holding what transformers does not know: the projections and the cross-attention blocks. `load_stream_encoder` puts it back together. `save_pretrained` calls it, so an export writes a stream encoder the way it writes any other encoder:

In [ ]:
#| export
def save_stream_encoder(enc:StreamEncoder, path) -> Path:
    "Write a stream encoder: transformers folders for the text and the side models, and a file for the projections and cross-attention blocks"
    from safetensors.torch import save_file
    path = Path(path); path.mkdir(parents=True, exist_ok=True)
    layers = encoder_layers(enc.text) if enc.xattn else []
    wrapped = {i: l for i, l in enumerate(layers) if getattr(l, "is_xattn", False)}
    try:
        for i, l in wrapped.items(): layers[i] = l.layer          # the text encoder as transformers knows it
        enc.text.save_pretrained(path / "text")
    finally:
        for i, l in wrapped.items(): layers[i] = l
    extra = {f"proj.{k}": v for k, v in enc.proj.state_dict().items()}
    for name, side in enc.sides.items():
        side.model.save_pretrained(path / "sides" / name)
        if side.proj is not None: extra |= {f"sides.{name}.proj.{k}": v for k, v in side.proj.state_dict().items()}
    for i, l in wrapped.items(): extra |= {f"xattn.{i}.{k}": v for k, v in l.block.state_dict().items()}
    if extra: save_file({k: v.detach().cpu().contiguous() for k, v in extra.items()}, str(path / "stream_encoder.safetensors"))
    meta = {"fusion": enc.fusion, "xattn": enc.xattn,
            "sides": {n: {"special_ids": s.special_ids, "proj": s.proj.dims if s.proj is not None else None} for n, s in enc.sides.items()}}
    (path / "stream_encoder.json").write_text(json.dumps(meta, indent=1))
    return path

def load_stream_encoder(path, dtype=torch.float32) -> StreamEncoder:
    "A stream encoder written by `save_stream_encoder`"
    from safetensors.torch import load_file
    from transformers.utils import logging as hf_logging
    path = Path(path); meta = json.loads((path / "stream_encoder.json").read_text())
    level = hf_logging.get_verbosity(); hf_logging.set_verbosity_error()      # a pooler a side model never had is not news
    try:
        sides = {n: SideEncoder(AutoModel.from_pretrained(path / "sides" / n, dtype=dtype), m["special_ids"], ProjMLP(m["proj"]) if m["proj"] else None)
                 for n, m in meta["sides"].items()}
        enc = StreamEncoder(AutoModel.from_pretrained(path / "text", dtype=dtype), sides)
    finally: hf_logging.set_verbosity(level)
    if meta.get("xattn"): add_cross_attention(enc, **meta["xattn"])
    sd = load_file(str(path / "stream_encoder.safetensors")) if (path / "stream_encoder.safetensors").exists() else {}
    part = lambda pre: {k[len(pre):]: v for k, v in sd.items() if k.startswith(pre)}
    for n, s in enc.sides.items():
        if s.proj is not None: s.proj.load_state_dict(part(f"sides.{n}.proj."))
    enc.proj.load_state_dict(part("proj."))
    for i, l in enumerate(encoder_layers(enc.text) if meta.get("xattn") else []):
        if getattr(l, "is_xattn", False): l.block.load_state_dict(part(f"xattn.{i}."))
    return enc.to(dtype)

In [ ]:
import tempfile

In [ ]:
for e in (se, xse):
    back = load_stream_encoder(e.save_pretrained(Path(tempfile.mkdtemp()) / "se")).eval()
    with torch.no_grad():
        a, b = e.eval()(**enc_in), back(**enc_in)
        test_close(b.last_hidden_state, a.last_hidden_state, eps=1e-5)
        test_close(b.stream_vectors["protein"], a.stream_vectors["protein"], eps=1e-5)
test_eq(back.xattn, xse.xattn)

## Merging adapters

At export the adapters are merged into the encoder (`merge_and_unload`), so a LoRA or MiCA run ships as a plain encoder and nothing downstream needs `peft`:

In [ ]:
#| export
def merged_encoder(model:EncoderDecisionModel) -> nn.Module:
    "The encoder with any adapters merged in (a copy; the model keeps its adapters)"
    enc = model.encoder
    if hasattr(enc, "merge_and_unload"): return copy.deepcopy(enc).merge_and_unload()
    return enc

In [ ]:
m_lora.eval()
with torch.no_grad():
    for p in m_lora.encoder.parameters():
        if p.requires_grad: p.add_(torch.randn_like(p) * 0.05)   # pretend it trained
    before = m_lora(**inputs)
    plain = EncoderDecisionModel(merged_encoder(m_lora), m_lora.head, spec).eval()
    test_close(plain(**inputs), before, eps=1e-4)
test_eq(any("lora_" in n for n, _ in plain.encoder.named_parameters()), False)

## Laya checkpoints

A Laya checkpoint is an encoder config, a tokenizer, `rl_agent_config.json` (head layers, budgets, temperatures) and one `model.safetensors` holding the encoder (`encoder.*`), the head (`head.layers.*`, `type_emb`, `scorer`), an action head and a temperature buffer. `load_laya` reads one into an `EncoderDecisionModel` plus its template and temperatures, so a Laya checkpoint is also a sysone model: `Decider.load("convaiinnovations/laya")` works, and `init_from` can warm-start from it. The action head has no counterpart in sysone and is dropped.

In [ ]:
#| export
def laya_temperatures(cfg:dict) -> dict:
    "Laya's per-type and per-bucket temperatures in sysone's form"
    t = {QTYPE_NAMES[i]: float(v) for i, v in enumerate(cfg.get("temperature", [1.0, 1.0, 1.0]))}
    return t | {k: float(v) for k, v in (cfg.get("temperature_by_options") or {}).items()}

def _laya_dir(src, subfolder=None, revision=None):
    if Path(src).exists(): return Path(src) / (subfolder or "")
    from huggingface_hub import snapshot_download
    pre = f"{subfolder}/" if subfolder else ""
    d = snapshot_download(src, revision=revision, allow_patterns=[pre + p for p in
                          ("rl_agent_config.json", "model.safetensors", "tokenizer/*", "encoder/*")])
    return Path(d) / (subfolder or "")

def _laya_parts(d:Path):
    "The config, encoder config and tokenizer of a Laya checkpoint folder"
    cfg = json.loads((d / "rl_agent_config.json").read_text())
    ecfg = AutoConfig.from_pretrained(d / "encoder") if (d / "encoder").exists() else AutoConfig.from_pretrained(cfg["encoder"])
    tok = AutoTokenizer.from_pretrained(d / "tokenizer") if (d / "tokenizer").exists() else AutoTokenizer.from_pretrained(cfg["encoder"])
    return cfg, ecfg, tok

def _laya_encoder(d:Path, ecfg, sd=None):
    "The encoder of a Laya checkpoint, from its `encoder.*` tensors"
    if sd is None:
        from safetensors.torch import load_file
        sd = load_file(str(d / "model.safetensors"))
    enc = AutoModel.from_config(ecfg, dtype=torch.float32)
    missing, _ = enc.load_state_dict({k[len("encoder."):]: v.float() for k, v in sd.items() if k.startswith("encoder.")}, strict=False)
    missing = [k for k in missing if k not in (getattr(enc, "_tied_weights_keys", None) or {}) and "rotary" not in k]
    if missing: raise ValueError(f"{d}: the checkpoint lacks encoder weights {missing[:3]}")
    return enc

def load_laya(src:str, subfolder:str|None=None, revision:str|None=None):
    "An `EncoderDecisionModel`, its tokenizer, template and temperatures from a Laya checkpoint"
    from safetensors.torch import load_file
    if subfolder is None: src, subfolder = split_repo(src)
    d = _laya_dir(src, subfolder, revision)
    cfg, ecfg, tok = _laya_parts(d)
    sd = load_file(str(d / "model.safetensors"))
    act = "act_head.2.weight" in sd and sd["act_head.2.weight"].shape[0] == 2        # a trained act head, not an export's constant one
    head = DecisionHead(ecfg.hidden_size, layers=cfg.get("head_layers", 2), act=act)
    head.load_state_dict({k: v.float() for k, v in from_laya_names(sd).items() if act or not k.startswith("act_head.")})
    template = RowTemplate.preset("laya", tokenizer=tok, max_len=cfg.get("max_len", 512), head_max_len=cfg.get("head_max_len", 192))
    spec = EncoderSpec(cfg["encoder"], ecfg, tok, template=template, source="laya-checkpoint")
    model = EncoderDecisionModel(_laya_encoder(d, ecfg, sd), head, spec).eval()
    return model, tok, template, laya_temperatures(cfg)

A Laya checkpoint is also an encoder source: `EncoderSpec.from_pretrained("convaiinnovations/laya")` reads its encoder config, tokenizer and budgets, and loads the encoder from its single weights file. This is how Laya's own fine-tuning notebook starts, from the released checkpoint's encoder and head, so reproducing it is `decision_learner(data, "convaiinnovations/laya", init_from="convaiinnovations/laya", train="full")`. A subfolder of a repo that bundles several checkpoints is written `owner/repo/subfolder`.

In [ ]:
#| export
def _laya_spec(id:str, revision:str|None=None, **kw) -> EncoderSpec:
    repo, sub = split_repo(id)
    d = _laya_dir(repo, sub, revision)
    cfg, ecfg, tok = _laya_parts(d)
    template = RowTemplate.preset("laya", tokenizer=tok, max_len=cfg.get("max_len", 512), head_max_len=cfg.get("head_max_len", 192))
    def loader(spec, dtype=None, **kw):
        enc = _laya_encoder(d, ecfg)
        return enc.to(dtype) if dtype is not None else enc
    return EncoderSpec(id, ecfg, tok, revision=revision, source="laya-checkpoint", template=template, loader=loader)

ENCODER_SOURCES.append(("laya", lambda id, revision=None: _is_laya(*split_repo(id)), _laya_spec))

Laya's runtime builds rows with its own rules and runs no sysone transforms, so the format is written only when the template is Laya's (with Laya's rule of keeping a conversation's end), the head is Laya's shape, and no transform runs at inference. `laya_state_dict` goes the other way: the full model under Laya's names, with a zeroed one-output action head (Laya's runtime then reports an action probability of 1) and the per-type temperatures in the buffer, which is what `laya.Agent` loads with `strict=True`. The export notebook writes it next to `rl_agent_config.json`.

In [ ]:
#| export
def laya_state_dict(model:EncoderDecisionModel, temperatures:dict|None=None, dtype=None) -> dict:
    "The model's weights under Laya's parameter names, loadable by `laya.Agent`"
    enc = merged_encoder(model)
    d = model.head.width
    sd = {f"encoder.{k}": v for k, v in enc.state_dict().items()}
    sd |= to_laya_names(model.head.state_dict())
    if not getattr(model.head, "act", False):    # Laya's format always has an act head: without one, a constant one
        sd |= {"act_head.0.weight": torch.zeros(256, d + 4), "act_head.0.bias": torch.zeros(256),
               "act_head.2.weight": torch.zeros(1, 256), "act_head.2.bias": torch.zeros(1)}
    t = temperatures or {}
    sd["temperature"] = torch.tensor([clamp_temperature(t.get(n, 1.0)) for n in ("choice", "score", "noul")])
    sd = {k: v.detach().to("cpu").contiguous() for k, v in sd.items()}
    return {k: v.to(dtype) if dtype is not None and v.is_floating_point() else v for k, v in sd.items()}

def is_laya_compatible(model:EncoderDecisionModel, template:RowTemplate, tfms=(), tokenizer=None) -> bool:
    "Whether an export can also be written in Laya's format: Laya's row and budgets, Laya's head, no inference transforms"
    t, h = template, model.head
    family = getattr(getattr(model.encoder, "config", None), "model_type", None)
    tok = getattr(tokenizer, "tokenizer", tokenizer)
    specials = (tok.cls_token, tok.sep_token, tok.sep_token, tok.mask_token) if tok is not None else ("[CLS]", "[SEP]", "[SEP]", "[MASK]")
    return (t.row == RowTemplate.preset("laya").row and t.option == "{mask} {text}" and t.option_sep == ""
            and (t.start, t.sep, t.end, t.mask) == specials and t.truncate == "auto"
            and t.option_tokens == 48 and h.scorer_kind in ("laya", "mlp") and h.nhead == max(1, h.width // 64)
            and h.readout == "anchor" and h.query is None
            and family == "modernbert" and not h.standardize and not any(not x.train_only for x in tfms))

Round trip on the fixture: the Laya-named state dict loads back into the same model, and the `laya` package's own `DecisionModel` computes the same logits from it:

In [ ]:
from laya.common import DecisionModel as LayaDecisionModel

In [ ]:
torch.manual_seed(1)
ours = EncoderDecisionModel(spec.load_encoder(), DecisionHead(spec.width), spec).eval()
sd = laya_state_dict(ours, {"choice": 1.3})
theirs = LayaDecisionModel(spec.load_encoder(), head_layers=2, n_act=1).eval()
theirs.load_state_dict(sd, strict=True)
with torch.no_grad():
    lt, _ = theirs(batch["input_ids"], batch["attention_mask"], batch["marker_pos"], batch["marker_mask"], batch["qtype"])
    test_close(ours(**inputs), lt, eps=1e-5)
test_eq(is_laya_compatible(ours, spec.template, tokenizer=tok), True)
test_eq(is_laya_compatible(ours, spec.template.replace(mask="[L]"), tokenizer=tok), False)

A head with an act head is written with it, and Laya's own model then computes the same act logits too, so an act head trained by sysone runs in Laya's runtime, and one of Laya's loads into sysone:

In [ ]:
torch.manual_seed(2)
ours_act = EncoderDecisionModel(spec.load_encoder(), DecisionHead(spec.width, act=True), spec).eval()
theirs_act = LayaDecisionModel(spec.load_encoder(), head_layers=2, n_act=2).eval()
theirs_act.load_state_dict(laya_state_dict(ours_act), strict=True)
with torch.no_grad():
    lt2, at2 = theirs_act(batch["input_ids"], batch["attention_mask"], batch["marker_pos"], batch["marker_mask"], batch["qtype"])
    ol, oa = ours_act(**inputs)
test_close(ol, lt2, eps=1e-5); test_close(oa, at2, eps=1e-5)
test_eq(is_laya_compatible(EncoderDecisionModel(ours.encoder, DecisionHead(spec.width, readout="span"), spec), spec.template, tokenizer=tok), False)

Written out as a Laya checkpoint folder, the fixture model is an encoder source and a head to warm-start from, and both come back exactly:

In [ ]:
import tempfile
from safetensors.torch import save_file

In [ ]:
ck = Path(tempfile.mkdtemp()) / "laya-ck"
ck.mkdir()
save_file(sd, str(ck / "model.safetensors"))
(ck / "rl_agent_config.json").write_text(json.dumps({"encoder": FIX, "head_layers": 2, "max_len": 256, "head_max_len": 96}))
ours.encoder.config.save_pretrained(ck / "encoder"); tok.save_pretrained(ck / "tokenizer")
ls = EncoderSpec.from_pretrained(str(ck))
test_eq((ls.source, ls.template.max_len, ls.template.head_max_len), ("laya-checkpoint", 256, 96))
warm = EncoderDecisionModel(ls.load_encoder(), DecisionHead(ls.width, init_from=str(ck)), ls).eval()
with torch.no_grad(): test_close(warm(**inputs), ours(**inputs), eps=1e-6)
test_eq(split_repo("convaiinnovations/laya/typed-decisions"), ("convaiinnovations/laya", "typed-decisions"))
test_eq(split_repo("answerdotai/ModernBERT-large"), ("answerdotai/ModernBERT-large", None))
test_eq(repr(checkpoint(str(ck))), f"LayaCheckpoint({str(ck)!r})")
test_fail(lambda: checkpoint(FIX), contains="is not a Laya checkpoint")

In [ ]:
#| eval: false
laya_model, laya_tok, laya_template, laya_temps = load_laya("convaiinnovations/laya", subfolder="typed-decisions")
laya_template, laya_temps

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()